# Evaluation Metrics — Classification (6/19–6/20/26)

---

*Codecademy — MLE Path: ML Fundamentals. Concise grad-student notes. Lesson complete ✅*

> **Scope of this lesson:** how to judge a *classifier* once it's trained. The single number "accuracy" is often a trap; this lesson builds the **confusion matrix** and the four metrics derived from it (accuracy, recall, precision, F1), then how to **choose** among them by which error costs more.

## Confusion Matrix — the 4 outcomes

> **Prerequisite:** measuring predictive power happens *after* splitting into **train / validation / test** — you compute these on the held-out evaluation set, never on training data. (See `Train_Validation_Test_Lesson.ipynb`.)

Pass the evaluation set's features through the trained model → list of **predictions**. Compare each to the **actual label**. Every comparison lands in one of **four buckets** (spam-classifier framing):

- **True Positive (TP):** predicted spam, *was* spam ✅
- **True Negative (TN):** predicted not-spam, *was* not-spam ✅
- **False Positive (FP):** predicted spam, was *not* spam ❌ (false alarm)
- **False Negative (FN):** predicted not-spam, *was* spam ❌ (miss)

Read each as *"<True/False> <what the model predicted>"*: **False Positive** = model said positive (spam) and was wrong.

### The confusion matrix (Codecademy layout)
Predicted classes = **columns**, actual classes = **rows**:

| | Predicted − | Predicted + |
|---|---|---|
| **Actual −** | TN | FP |
| **Actual +** | FN | TP |

Everything below (accuracy, recall, precision, F1) is just arithmetic on these four counts.

## Accuracy — the naive default

Correctly classified predictions (TP **and** TN) over *all* predictions:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} = \frac{\text{correct}}{\text{total}}$$

- For our exercise data: $\frac{TP+TN}{\text{total}} = \frac{3+0}{10} = 0.30$ — a weak classifier.
- Intuitive, but **misleading on imbalanced data**. *Class-imbalance trap:* if 99% of emails are not-spam, a model that always predicts "not spam" scores **99% accuracy** while catching *zero* spam. Looks great, useless.
- Accuracy weights every error equally and ignores *which* class you fail on → motivates recall & precision next.

In [1]:
# Accuracy exercise -- your solution (helper fn). Self-contained so this cell
# runs on its own; the four counts come from the exercise data further down.
true_positives, true_negatives, false_positives, false_negatives = 3, 0, 3, 4

def Acc(TP, TN, FP, FN):
    return (TP + TN) / (TP + TN + FP + FN)   # correct / total

accuracy = Acc(true_positives, true_negatives, false_positives, false_negatives)
print(accuracy)   # 0.3

0.3


## Recall — "of the real positives, how many did we catch?"

**Why accuracy isn't enough (the motivating example):** imagine a spam classifier for *top-secret government inboxes* that almost never receive spam. A lazy model that **always predicts "not spam"** scores near-perfect **accuracy** — yet it never finds the one thing we care about (the spam). High accuracy, zero usefulness.

$$\text{Recall} = \frac{TP}{TP + FN}$$

- Recall = ratio of **correct positive predictions** to **all actual positives**. For the spam case: correctly-flagged spam ÷ every email that *was* actually spam.
- Denominator = all actual positives → answers *how complete is the catch?*
- The always-"not spam" classifier produces **no true positives**, so its **recall = 0** — exactly the failure accuracy hid.
- *(Standard aside, not in the lesson: recall is also called **sensitivity** / **true positive rate**.)*

In [2]:
# Recall exercise -- self-contained so the cell runs on its own.
actual    = [1, 0, 0, 1, 1, 1, 0, 1, 1, 1]
predicted = [0, 1, 1, 1, 1, 0, 1, 0, 1, 0]

true_positives = true_negatives = false_positives = false_negatives = 0
for i in range(len(predicted)):
    if actual[i] == 1 and predicted[i] == 1: true_positives  += 1
    if actual[i] == 0 and predicted[i] == 0: true_negatives  += 1
    if actual[i] == 0 and predicted[i] == 1: false_positives += 1
    if actual[i] == 1 and predicted[i] == 0: false_negatives += 1

# NOTE: define Recall OUTSIDE the loop. In the exercise paste it was indented
# inside the for-loop (redefined every pass) -- works, but wrong placement.
def Recall(TP, FN):
    return TP / (TP + FN)

recall = Recall(true_positives, false_negatives)
print(f"Recall = {recall}")   # 3/(3+4) = 0.42857...

Recall = 0.42857142857142855


## Precision — "of our positive calls, how many were right?"

**Why recall isn't enough either** (no metric is perfect): a classifier that **always predicts "spam"** catches every real spam → **recall = 1**, but it's useless — it also drowns you in false alarms. The mirror image of the always-"not spam" model from the recall section.

$$\text{Precision} = \frac{TP}{TP + FP}$$

- Precision = ratio of **correct positive predictions** to **all positive predictions the model made**. For spam: correctly-flagged spam ÷ everything the model labeled spam (right *or* wrong).
- Denominator = all **predicted** positives → answers *how trustworthy is a positive call?*
- The always-"spam" classifier has tons of **false positives**, so precision is very low even though recall is perfect.
- Precision vs recall is a **trade-off**: being stricter about calling something positive raises precision but lowers recall, and vice versa → motivates F1 next.

In [3]:
# Precision exercise -- self-contained so the cell runs on its own.
actual    = [1, 0, 0, 1, 1, 1, 0, 1, 1, 1]
predicted = [0, 1, 1, 1, 1, 0, 1, 0, 1, 0]

true_positives = true_negatives = false_positives = false_negatives = 0
for i in range(len(predicted)):
    if actual[i] == 1 and predicted[i] == 1: true_positives  += 1
    if actual[i] == 0 and predicted[i] == 0: true_negatives  += 1
    if actual[i] == 0 and predicted[i] == 1: false_positives += 1
    if actual[i] == 1 and predicted[i] == 0: false_negatives += 1

precision = true_positives / (true_positives + false_positives)
print(precision)   # 3/(3+3) = 0.5

0.5


## F1 Score — one number that balances the two

Often we want to judge a model on **precision and recall together**. The F1-score combines them into a single number via their **harmonic mean** (a kind of averaging):

$$F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

**Why harmonic, not arithmetic?** We want F1 to be **low whenever *either* precision or recall is low**. Worked example from the lesson — a classifier with **recall = 1, precision = 0.02** (great catch rate, but almost every positive call is wrong):

- Arithmetic mean: $\frac{1 + 0.02}{2} = 0.51$ → **misleadingly high** for such a broken model.
- Harmonic mean (F1): $2 \cdot \frac{1 \times 0.02}{1 + 0.02} = 0.039$ → **honest** — reflects how bad the precision is.

The harmonic mean is dragged toward the *smaller* of the two, so F1 is only high when **both** precision and recall are high. Ranges 0→1.

In [4]:
# F1-score exercise -- self-contained; F1 built from the recall & precision helpers.
actual    = [1, 0, 0, 1, 1, 1, 0, 1, 1, 1]
predicted = [0, 1, 1, 1, 1, 0, 1, 0, 1, 0]

true_positives = true_negatives = false_positives = false_negatives = 0
for i in range(len(predicted)):
    if actual[i] == 1 and predicted[i] == 1: true_positives  += 1
    if actual[i] == 0 and predicted[i] == 0: true_negatives  += 1
    if actual[i] == 0 and predicted[i] == 1: false_positives += 1
    if actual[i] == 1 and predicted[i] == 0: false_negatives += 1

def Recall(TP, FN): return TP / (TP + FN)
def Prec(TP, FP):   return TP / (TP + FP)

def F1_score(TP, FP, FN):
    r = Recall(TP, FN)
    p = Prec(TP, FP)
    return (2 * p * r) / (p + r)   # harmonic mean of precision & recall

f_1 = F1_score(true_positives, false_positives, false_negatives)
print("F1-score = ", f_1)   # 2*(0.5*0.4286)/(0.5+0.4286) = 0.46154...

F1-score =  0.4615384615384615


## Exercise — count the four outcomes & build the matrix

*Codecademy exercise.* `actual` = true labels (1 = spam, 0 = not spam); `predicted` = the classifier's calls.
Steps: (1) init four counters to 0 → (2) loop, `+1` to the matching bucket per email → (3) print them → (4) build the confusion matrix with sklearn.

In [5]:
from sklearn.metrics import confusion_matrix

actual    = [1, 0, 0, 1, 1, 1, 0, 1, 1, 1]
predicted = [0, 1, 1, 1, 1, 0, 1, 0, 1, 0]

# initializing confusion-matrix elements
true_positives  = 0
true_negatives  = 0
false_positives = 0
false_negatives = 0

for i in range(len(predicted)):
    if actual[i] == 1 and predicted[i] == 1:   # predicted spam, was spam
        true_positives += 1
    if actual[i] == 0 and predicted[i] == 0:   # predicted not-spam, was not-spam
        true_negatives += 1
    if actual[i] == 0 and predicted[i] == 1:   # predicted spam, was not-spam (false alarm)
        false_positives += 1
    if actual[i] == 1 and predicted[i] == 0:   # predicted not-spam, was spam (miss)
        false_negatives += 1

print(true_positives)    # 3
print(true_negatives)    # 0
print(false_positives)   # 3
print(false_negatives)   # 4

# scikit-learn builds it for you -> layout [[TN, FP], [FN, TP]]
conf_matrix = confusion_matrix(actual, predicted)
print(conf_matrix)       # [[0 3]
                         #  [4 3]]

3
0
3
4
[[0 3]
 [4 3]]


## The scikit-learn way (what you'll actually use)

In [6]:
# Final exercise -- sklearn does all four for you. Each takes (actual, predicted).
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

actual    = [1, 0, 0, 1, 1, 1, 0, 1, 1, 1]
predicted = [0, 1, 1, 1, 1, 0, 1, 0, 1, 0]

print(accuracy_score(actual, predicted))    # 0.3
print(recall_score(actual, predicted))      # 0.42857...
print(precision_score(actual, predicted))   # 0.5
print(f1_score(actual, predicted))          # 0.46153...

# Bonus: classification_report gives all of them per class in one shot.
from sklearn.metrics import classification_report
print("\n", classification_report(actual, predicted))

0.3
0.42857142857142855
0.5
0.46153846153846156

               precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.50      0.43      0.46         7

    accuracy                           0.30        10
   macro avg       0.25      0.21      0.23        10
weighted avg       0.35      0.30      0.32        10



> **sklearn `confusion_matrix` layout gotcha:** rows = actual, cols = predicted, ordered `[0, 1]`. So it's `[[TN, FP], [FN, TP]]` — *not* the textbook layout above. Always sanity-check which corner is which.

---
## Q & A (captured as I go)

*Questions I posed during the lesson + answers, folded in per sublesson.*

**Q (Recall): Give another example so I actually get recall.**

**A — Cancer screening.** 100 patients truly have cancer. Test correctly flags 80 (TP), misses 20 (FN).
$\text{Recall} = \frac{TP}{TP+FN} = \frac{80}{100} = 0.80$ → *of everyone who really had it, we caught 80%.*

Key intuition:
- Recall only looks at the **actual positives** and asks *how many did we catch?*
- Its enemy is the **False Negative** (the miss) — every miss drags recall down.
- It **ignores false positives** entirely (that's precision's concern). So flag *everyone* positive → recall = 1.0 but useless → never report recall alone.
- **Fishing-net picture:** lake has 100 fish you want; your net lands 80 → recall 80/100. Recall doesn't care about the seaweed you also pulled up (precision does) — only *what share of the real fish you caught.*
- Mnemonic: **Re-call = did I re-call (retrieve) all the real ones?**

---
## Choosing a metric — the review

**There is no perfect metric.** Which one (accuracy, precision, recall, F1, or something else) you optimize depends entirely on the **question the classification problem is asking** — specifically, *which kind of error hurts more.*

**Worked judgment call — the spam filter:** here we want **high precision** and can **tolerate lower recall**. Why? The costs are asymmetric:
- A **false positive** (a real, important email wrongly sent to spam) is **bad** — you might miss something that matters.
- A **false negative** (a spam email slipping into the inbox) is **mildly annoying** — easy to delete.

So we'd rather let a little spam through than risk burying a real message → optimize **precision**. (Contrast: cancer screening flips this — a missed case is catastrophic, so you optimize **recall** instead.)

**Rule of thumb:** decide which error is more costly → pick the metric that punishes that error.
- Misses (FN) hurt most → **recall**
- False alarms (FP) hurt most → **precision**
- Need both → **F1**

And in practice, **scikit-learn computes all of these for you** (see the cell above).

---
## TL;DR

- Everything comes from the **confusion matrix**: TP / FP / FN / TN.
- **Accuracy** = overall correctness; lies on imbalanced data.
- **Recall** = `TP/(TP+FN)` — caught positives; maximize when *misses* are costly.
- **Precision** = `TP/(TP+FP)` — trustworthy positives; maximize when *false alarms* are costly.
- **F1** = harmonic mean of the two; high only when *both* are high. Use for imbalanced classes.
- **No perfect metric** — choose by which error costs more: misses→recall, false alarms→precision, both→F1 (e.g. spam filter → precision; cancer screening → recall).
- In practice: `from sklearn.metrics import ...` + `classification_report` for all of it at once.